# B4 — Entry Z-Score Violin by Direction

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
import matplotlib.patches as mpatches
all_trades = []
for wk, wm in WINDOWS_META.items():
    rk = wm['result_key']
    t = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'trades.parquet')
    t['window'] = wk
    all_trades.append(t)
df = pd.concat(all_trades, ignore_index=True)

fig, axes = plt.subplots(1, 4, figsize=(16, 5), sharey=True)
for ax, wk in zip(axes, WINDOWS_META.keys()):
    sub = df[df['window'] == wk]
    for j, direction in enumerate(['Long', 'Short']):
        data = sub[sub['dir_label'] == direction]['entry_z'].dropna().values
        if len(data) == 0:
            continue
        parts = ax.violinplot([data], positions=[j], showmedians=True, widths=0.6)
        color = '#2ecc71' if direction == 'Long' else '#e74c3c'
        for pc in parts['bodies']:
            pc.set_facecolor(color); pc.set_alpha(0.7)
    ax.axhline(2.5, color='grey', ls='--', lw=0.8)
    ax.axhline(-2.5, color='grey', ls='--', lw=0.8)
    ax.set_xticks([0,1]); ax.set_xticklabels(['Long','Short'])
    ax.set_title(wk)
axes[0].set_ylabel('Entry Z-Score')
fig.suptitle('Entry Z-Score Distribution by Direction — All Windows', fontsize=13)
save_fig(fig, 'B4_entry_zscore_violin.png')
